# 📦 Notebook 04 — Product & Seller Analysis

**Goal:** Identify top-performing products, categories, and sellers. Understand pricing, review patterns, and opportunities.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

DATA_PATH = '../data/'

orders       = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv',
                           parse_dates=['order_purchase_timestamp'])
order_items  = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
products     = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
reviews      = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
sellers      = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
category_map = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('Data loaded ✅')

## 1. Build Product Master

In [ ]:
df = order_items.merge(products[['product_id','product_category_name',
                                  'product_weight_g','product_photos_qty']],
                        on='product_id', how='left')
df = df.merge(category_map, on='product_category_name', how='left')
df = df.merge(orders[['order_id','order_status','order_purchase_timestamp']],
              on='order_id', how='left')
df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')
df = df[df['order_status'] == 'delivered']

print(f'Product master shape: {df.shape}')
df.head(3)

## 2. Top Categories — Volume vs Revenue vs Rating

In [ ]:
cat_summary = df.groupby('product_category_name_english').agg(
    units_sold   = ('order_id', 'count'),
    total_revenue = ('price', 'sum'),
    avg_price    = ('price', 'mean'),
    avg_rating   = ('review_score', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)

top15 = cat_summary.head(15)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, col, title, color in zip(
    axes,
    ['units_sold','total_revenue','avg_rating'],
    ['Units Sold','Total Revenue (R$)','Avg Rating'],
    ['steelblue','seagreen','coral']
):
    data = top15.sort_values(col, ascending=True)
    ax.barh(data['product_category_name_english'], data[col], color=color, alpha=0.85)
    ax.set_title(f'Top 15 Categories\nby {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel(title)

plt.tight_layout()
plt.savefig('../data/category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Price Distribution by Category

In [ ]:
top8_cats = cat_summary.head(8)['product_category_name_english'].tolist()
df_top8 = df[df['product_category_name_english'].isin(top8_cats)]

fig, ax = plt.subplots(figsize=(14, 6))
df_top8.boxplot(column='price', by='product_category_name_english',
                ax=ax, showfliers=False,
                patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
ax.set_title('Price Distribution — Top 8 Categories', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Price (R$)')
plt.suptitle('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../data/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Top 10 Sellers by Revenue

In [ ]:
seller_perf = df.groupby('seller_id').agg(
    total_revenue = ('price', 'sum'),
    total_orders  = ('order_id', 'nunique'),
    avg_rating    = ('review_score', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)

top10_sellers = seller_perf.head(10)
top10_sellers['seller_label'] = ['Seller ' + str(i+1) for i in range(10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(top10_sellers['seller_label'], top10_sellers['total_revenue'],
            color=sns.color_palette('viridis', 10))
axes[0].set_title('Top 10 Sellers by Revenue', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Seller')
axes[0].set_ylabel('Revenue (R$)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].scatter(top10_sellers['total_orders'], top10_sellers['avg_rating'],
                s=top10_sellers['total_revenue']/500,
                c=range(10), cmap='viridis', alpha=0.8)
for _, row in top10_sellers.iterrows():
    axes[1].annotate(row['seller_label'],
                     (row['total_orders'], row['avg_rating']),
                     fontsize=8, ha='center')
axes[1].set_title('Seller Performance: Orders vs Rating\n(bubble size = revenue)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Orders')
axes[1].set_ylabel('Avg Review Score')

plt.tight_layout()
plt.savefig('../data/seller_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Photo Count vs Sales Volume

In [ ]:
photo_impact = df.groupby('product_photos_qty').agg(
    units_sold  = ('order_id', 'count'),
    avg_price   = ('price', 'mean'),
    avg_rating  = ('review_score', 'mean')
).reset_index().dropna()

photo_impact = photo_impact[photo_impact['product_photos_qty'] <= 10]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(photo_impact['product_photos_qty'], photo_impact['units_sold'],
            color='steelblue')
axes[0].set_title('Number of Product Photos vs Units Sold', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Photos')
axes[0].set_ylabel('Units Sold')

axes[1].plot(photo_impact['product_photos_qty'], photo_impact['avg_rating'],
             marker='o', color='coral', linewidth=2)
axes[1].set_title('Number of Product Photos vs Avg Rating', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Photos')
axes[1].set_ylabel('Average Review Score')

plt.tight_layout()
plt.savefig('../data/photo_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Products with more photos tend to sell more and get better ratings!')

---
## ✅ Product Analysis Summary

| Finding | Insight |
|---------|----------|
| Top revenue category | Bed, Bath & Table |
| Highest rated category | Books & media |
| Photo count impact | More photos = more sales + better ratings |
| Top sellers | High volume but varying quality scores |

🎉 **You've completed all 4 analysis notebooks!**

Next steps:
- Load `rfm_segments.csv` into Power BI / Tableau for dashboarding
- Review `sql/queries.sql` for executive reporting
- Update `README.md` with your own name and push to GitHub!